# HACKIA 2026
## Phase 3 – Description de scenes
UMONS, 2025–2026

Auteurs :  
- Ilario Strazzeri
- Quentin Campeol

# Imports

In [1]:
import torch
import json
import sys

import torchvision.transforms.functional as _F_func
sys.modules["torchvision.transforms.functional_tensor"] = _F_func

import cv2
import numpy as np

from torchvision.transforms._transforms_video import (
    CenterCropVideo,
    NormalizeVideo,
)
from torchvision.transforms import Compose, Lambda

# ✅ Importer les CLASSES (pas les fonctions) depuis pytorchvideo.transforms
from pytorchvideo.transforms import (
    ApplyTransformToKey,
    ShortSideScale,
    UniformTemporalSubsample,
    UniformCropVideo,
)

from pytorchvideo.data.encoded_video import EncodedVideo
from torchvision.transforms._functional_video import normalize

from typing import Dict

/home/hackia_group/hackia/lib/python3.12/site-packages/torchvision/transforms/_functional_video.py:6: UserWarning: The 'torchvision.transforms._functional_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms.functional' module instead.
  warnings.warn(
/home/hackia_group/hackia/lib/python3.12/site-packages/torchvision/transforms/_transforms_video.py:22: UserWarning: The 'torchvision.transforms._transforms_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms' module instead.
  warnings.warn(


# Chargement du modèle

In [2]:
# Device on which to run the model
# Set to cuda to load on GPU
device = "cuda"

# Pick a pretrained model and load the pretrained weights
model_name = "slowfast_r50"
model = torch.hub.load("facebookresearch/pytorchvideo", model=model_name, pretrained=True)

# Set to eval mode and move to desired device
model = model.to(device)
model = model.eval()


Using cache found in /home/hackia_group/.cache/torch/hub/facebookresearch_pytorchvideo_main


# Setup labels

In [3]:
!wget https://dl.fbaipublicfiles.com/pyslowfast/dataset/class_names/kinetics_classnames.json

--2026-05-16 14:52:25--  https://dl.fbaipublicfiles.com/pyslowfast/dataset/class_names/kinetics_classnames.json
Résolution de dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)… 18.239.208.3, 18.239.208.114, 18.239.208.8, ...
Connexion à dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|18.239.208.3|:443… connecté.
requête HTTP transmise, en attente de la réponse… 200 OK
Taille : 10326 (10K) [text/plain]
Enregistre : ‘kinetics_classnames.json.1’

kinetics_classnames 100%[===================>]  10.08K  --.-KB/s    ds 0s      

2026-05-16 14:52:25 (43.3 MB/s) - ‘kinetics_classnames.json.1’ enregistré [10326/10326]



In [5]:
with open("kinetics_classnames.json", "r") as f:
    kinetics_classnames = json.load(f)

# Create an id to label name mapping
kinetics_id_to_classname = {}
for k, v in kinetics_classnames.items():
    kinetics_id_to_classname[v] = str(k).replace('"', "")

# SlowFast transform

In [ ]:
side_size = 256
mean = [0.45, 0.45, 0.45]
std = [0.225, 0.225, 0.225]
crop_size = 256
num_frames = 32
sampling_rate = 2
frames_per_second = 30
alpha = 4

class PackPathway(torch.nn.Module):
    """
    Transform for converting video frames as a list of tensors.
    """
    def __init__(self):
        super().__init__()

    def forward(self, frames: torch.Tensor):
        fast_pathway = frames
        # Perform temporal sampling from the fast pathway.
        slow_pathway = torch.index_select(
            frames,
            1,
            torch.linspace(
                0, frames.shape[1] - 1, frames.shape[1] // alpha
            ).long(),
        )
        frame_list = [slow_pathway, fast_pathway]
        return frame_list

transform = ApplyTransformToKey(
    key="video",
    transform=Compose(
        [
            UniformTemporalSubsample(num_frames), 
            Lambda(lambda x: x / 255.0),
            NormalizeVideo(mean, std),
            ShortSideScale(size=side_size), 
            CenterCropVideo(crop_size),
            PackPathway()
        ]
    ),
)

In [8]:
# The duration of the input clip is also specific to the model.
clip_duration = (num_frames * sampling_rate)/frames_per_second

# Load the example video
video_path = "/home/hackia_group/Phase3/video/Fire_video.mp4"

# Select the duration of the clip to load by specifying the start and end duration
# The start_sec should correspond to where the action occurs in the video
start_sec = 0
end_sec = start_sec + clip_duration

# Initialize an EncodedVideo helper class
video = EncodedVideo.from_path(video_path)

# Load the desired clip
video_data = video.get_clip(start_sec=start_sec, end_sec=end_sec)

# Apply a transform to normalize the video input
video_data = transform(video_data)

# Move the inputs to the desired device
inputs = video_data["video"]
inputs = [i.to(device)[None, ...] for i in inputs]

# Classify the actions 

In [9]:
# Pass the input clip through the model
preds = model(inputs)

In [10]:
# Get the predicted classes
post_act = torch.nn.Softmax(dim=1)
preds = post_act(preds)
pred_classes = preds.topk(k=5).indices

# Map the predicted classes to the label names
pred_class_names = [kinetics_id_to_classname[int(i)] for i in pred_classes[0]]
print("Predicted labels: %s" % ", ".join(pred_class_names))

Predicted labels: playing paintball, riding mountain bike, archery, throwing axe, driving tractor


# QWEN3

In [2]:
! pip install git+https://github.com/huggingface/transformers
! pip install qwen-vl-utils[decord]
# pip install transformers==4.57.0 # currently, V4.57.0 is not released


  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-oyp0kqja
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-oyp0kqja
  Resolved https://github.com/huggingface/transformers to commit 3ef278124e47832f34406ca3ca85bc50ad8b79bb
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-4B-Instruct", torch_dtype="auto", device_map="auto"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-4B-Instruct")

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": "/home/hackia_group/Phase3/video/Fire_video.mp4",
                "max_pixels": 360 * 420,
                "fps": 1.0,
            },
            {"type": "text", "text": "Describe this video."},
        ],
    }
]

text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs, video_kwargs = process_vision_info(
    messages, return_video_kwargs=True
)

# ✅ Fix : aplatir les valeurs de video_kwargs si elles sont des listes
video_kwargs_fixed = {
    k: (v[0] if isinstance(v, list) and len(v) == 1 else v)
    for k, v in video_kwargs.items()
}


inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    return_tensors="pt",
    **video_kwargs_fixed,
)
inputs = inputs.to(model.device)  # ← .to(model.device) complet

generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids):]
    for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)
print(output_text[0])

/home/hackia_group/hackia/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 713/713 [00:01<00:00, 515.01it/s] 
qwen-vl-utils using decord to read video.
[transformers] Qwen3VL requires frame timestamps to construct prompts, but the `fps` of the input video could not be inferred. Probably `video_metadata` was missing from inputs and you passed pre-sampled frames. Defaulting to `fps=24`. Please provide `video_metadata` for more accurate results.


The video is a tutorial on how to build a fire for camping, presented by REI Co-op. It begins with an overview of the campsite, showing a fire pit surrounded by rocks and a tent in the background. The video then transitions to a close-up of a fire being built, with people roasting marshmallows and hot dogs over the flames. The camera pans to show the campsite, where people are sitting around the fire, enjoying their time. The video then shifts to a close-up of the fire pit, with the words "SAFETY TIPS" appearing on the screen. The camera then shows a person


In [6]:
output_text

['The video is a tutorial on how to build a fire for camping, presented by REI Co-op. It begins with an overview of the campsite, showing a fire pit surrounded by rocks and a tent in the background. The video then transitions to a close-up of a fire being built, with people roasting marshmallows and hot dogs over the flames. The camera pans to show the campsite, where people are sitting around the fire, enjoying their time. The video then shifts to a close-up of the fire pit, with the words "SAFETY TIPS" appearing on the screen. The camera then shows a person']